In [ ]:
                                 

import pandas as pd

from sklearn.metrics import roc_auc_score

from sklearn.metrics import precision_recall_curve, average_precision_score

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score, precision_score

import seaborn as sns

import matplotlib.pyplot as plt

from sklearn.preprocessing import normalize

import random

from Bio import SeqIO

import os 


In [ ]:
import pandas as pd

from sklearn.metrics import roc_auc_score

from sklearn.metrics import precision_recall_curve, average_precision_score

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score, precision_score

import seaborn as sns

import matplotlib.pyplot as plt

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.metrics import confusion_matrix

from sklearn.preprocessing import normalize


In [ ]:
def parse_fasta_to_dataframe(fasta_path):

    records = []

    with open(fasta_path, 'r') as file:

        name, tag, seq_lines = None, None, []

        for line in file:

            line = line.strip()

            if line.startswith('>'):

                if name:

                    sequence = ''.join(seq_lines)

                    records.append({'name': name, 'tag': tag, 'sequence': sequence})

                        

                header = line[1:]           

                                               

                if '_' in header:

                                  

                    last_underscore_index = header.rfind('_')

                    name = header[:last_underscore_index]                  

                    tag_str = header[last_underscore_index + 1:]                 

                    try:

                        tag = int(tag_str)             

                    except ValueError:

                        tag = None                 

                else:

                    name = header

                    tag = None

                seq_lines = []

            else:

                seq_lines.append(line)

                  

        if name:

            sequence = ''.join(seq_lines)

            records.append({'name': name, 'tag': tag, 'sequence': sequence})

    

    return pd.DataFrame(records)



In [ ]:
bootstrap_dir = 'circExor/benchmark_evaluation/exoGRU/bootstrap_results'

all_evaluate_dfs = []                    

for i in range(1, 6):

    fasta_path = os.path.join(bootstrap_dir, f'bootstrap_set_{i}.fasta')

    csv_path = os.path.join(bootstrap_dir, f'bootstrap_set_{i}_pred.csv')

    

                        

    df = parse_fasta_to_dataframe(fasta_path)

    pred = pd.read_csv(csv_path)

    

        

    evaluate_df = pd.concat([df, pred], axis=1)

    

              

    evaluate_df['prediction'] = (evaluate_df['prob'] >= 0.5).astype(int)

    all_evaluate_dfs.append(evaluate_df)                 

                                          

    globals()[f'evaluate_df_{i}'] = evaluate_df

    

    print(f"Successfully loaded and merged evaluate_df_{i}")

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, average_precision_score, matthews_corrcoef

auroc = roc_auc_score(evaluate_df["tag"], evaluate_df["prediction"])

auprc = average_precision_score(evaluate_df["tag"], evaluate_df["prediction"])

accuracy = accuracy_score(evaluate_df['tag'], evaluate_df['prediction'])

f1 = f1_score(evaluate_df['tag'], evaluate_df['prediction'])

mcc = matthews_corrcoef(evaluate_df['tag'], evaluate_df['prediction'])

print(f"AUROC: {auroc:.4f}")

print(f"AUPRC: {auprc:.4f}")

print(f"Accuracy: {accuracy:.4f}")

print(f"F1 Score: {f1:.4f}")

print(f"MCC: {mcc:.4f}")


In [ ]:


merged_evaluate_df = pd.concat(all_evaluate_dfs, ignore_index=True)

cm = confusion_matrix(merged_evaluate_df['tag'], merged_evaluate_df['prediction'])

                   

plt.figure(figsize=(6, 5))

ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 

                 xticklabels=['Celluar', 'EV'], yticklabels=['Celluar', 'EV'],

                 annot_kws={'size': 16, 'weight': 'bold'},                  

                 cbar_kws={'shrink': 0.8})                   

plt.title('exoGRU', fontsize=18, fontweight='bold')

plt.ylabel('Acurate label', fontsize=16)

plt.xlabel('Predicted label', fontsize=16)

plt.xticks(fontsize=15, fontweight='bold')

plt.yticks(fontsize=15, fontweight='bold')

                       

cbar = ax.collections[0].colorbar

cbar.ax.tick_params(labelsize=14)                      

plt.show()


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, average_precision_score, matthews_corrcoef, confusion_matrix

import pandas as pd

import numpy as np

           

bootstrap_results = {

    'Bootstrap': [],

    'AUROC': [],

    'AUPRC': [],

    'Accuracy': [],

    'Sensitivity': [],

    'Specificity': [],

    'F1 Score': [],

    'MCC': []

}

all_evaluate_dfs = []

for i in range(1, 6):

    df_eval = globals()[f'evaluate_df_{i}']

    all_evaluate_dfs.append(df_eval)

    

                

    y_true = df_eval['tag'].astype(int)

    y_pred = df_eval['prediction'].astype(int)

    

                                       

    y_prob = df_eval['prob'] 

    

          

    auroc = roc_auc_score(y_true, y_prob)

    auprc = average_precision_score(y_true, y_prob)

    acc = accuracy_score(y_true, y_pred)

    f1 = f1_score(y_true, y_pred)

    mcc = matthews_corrcoef(y_true, y_pred)

    

                                         

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    

            

    bootstrap_results['Bootstrap'].append(i)

    bootstrap_results['AUROC'].append(auroc)

    bootstrap_results['AUPRC'].append(auprc)

    bootstrap_results['Accuracy'].append(acc)

    bootstrap_results['Sensitivity'].append(sensitivity)

    bootstrap_results['Specificity'].append(specificity)

    bootstrap_results['F1 Score'].append(f1)

    bootstrap_results['MCC'].append(mcc)

                  

df_metrics = pd.DataFrame(bootstrap_results)

print("=== Metrics for each bootstrap run ===")

print(df_metrics.to_string(index=False))

print("\n")


In [ ]:
import matplotlib.pyplot as plt

import numpy as np

import scipy.stats as st

           

metrics = ['AUROC', 'AUPRC', 'Accuracy', 'F1 Score', 'MCC']

models_names = ['exoGRU']                 

                                       

model_data_dict = {

    'exoGRU': np.array([

        [0.460509, 0.504242, 0.4625, 0.505747, -0.079071],

        [0.516598, 0.490370, 0.5075, 0.547126,  0.020419],

        [0.478825, 0.499320, 0.4850, 0.529680, -0.030557],

        [0.470647, 0.527084, 0.4900, 0.542601, -0.032102],

        [0.507435, 0.500000, 0.4800, 0.490196, -0.034986]

    ])

}

             

warm_colors = ['#F8D7B4', '#F0C9A8', '#E8BB99', '#D89F7B', '#C8835D', '#B05930']

         

model_colors = [warm_colors[3]] 

        

fig, ax = plt.subplots(figsize=(10, 6))

n_metrics = len(metrics)

n_models = len(models_names)

                   

bar_width = 0.8 / (n_models + 0.5) if n_models > 1 else 0.4

x = np.arange(n_metrics)

max_y = 0                 

min_y = 0                          

              

for i, model_name in enumerate(models_names):

    data = model_data_dict[model_name]

    

                             

    means = np.mean(data, axis=0)

    stds = np.std(data, axis=0, ddof=1)

    n = data.shape[0]

    cis = st.t.ppf(0.975, n-1) * (stds / np.sqrt(n))

    

             

    offset = (i - n_models/2 + 0.5) * bar_width

    

                    

    bars = ax.bar(x + offset, means, bar_width, 

                  label=model_name, color=model_colors[i], 

                  edgecolor='black', linewidth=1.2, alpha=0.85,

                  yerr=cis, capsize=8, error_kw={'linewidth': 1.5, 'capthick': 1.5})

    

                

    for j, bar in enumerate(bars):

        height = means[j]

                              

        if height >= 0:

            current_top = height + cis[j]

            y_pos = current_top + 0.015

            va = 'bottom'

            if current_top > max_y: max_y = current_top

        else:

            current_bottom = height - cis[j]

            y_pos = current_bottom - 0.015

            va = 'top'

            if current_bottom < min_y: min_y = current_bottom

        ax.text(bar.get_x() + bar.get_width()/2., y_pos,

                f'{height:.3f}', ha='center', va=va, fontsize=13, fontweight='bold')

        

ax.set_xticks(x)

ax.set_xticklabels(metrics, fontsize=16, fontweight='bold')

ax.tick_params(axis='y', labelsize=14)

                               

ax.set_ylim(min_y * 1.5 if min_y < 0 else 0, max_y * 1.15) 

               

ax.axhline(0, color='black', linewidth=1)

ax.grid(axis='y', alpha=0.3, linestyle='--')

ax.set_ylabel('Score', fontsize=16, fontweight='bold')

ax.set_title(f'{models_names[0]} Performance (5 Bootstraps with 95% CI)', fontsize=18, fontweight='bold')

             

ax.legend(fontsize=14, loc='upper right')

      

plt.tight_layout()

plt.show()
